## 4. Logistic regression model
Quantify each driver's *independent* contribution to churn — i.e. its effect holding all other variables constant.

In [8]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/Churn_Analysis.csv').copy()
model_df = df.copy()

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### 4.1 Prepare features
Drop non-predictive columns and one-hot encode categorical variables.

In [9]:
model_df = model_df.drop(columns=["customerID", "Churn", "Churn_binary", "Tenure_Tier"])

model_df = pd.get_dummies(model_df, drop_first=True)

print(f"Columns:\n{model_df.columns}")
print(f"\nRows: {model_df.shape[0]}, and columns: {model_df.shape[1]}\n")
print(model_df.head(4))

Columns:
Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges',
       'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes',
       'MultipleLines_No phone service', 'MultipleLines_Yes',
       'InternetService_Fiber optic', 'InternetService_No',
       'OnlineSecurity_No internet service', 'OnlineSecurity_Yes',
       'OnlineBackup_No internet service', 'OnlineBackup_Yes',
       'DeviceProtection_No internet service', 'DeviceProtection_Yes',
       'TechSupport_No internet service', 'TechSupport_Yes',
       'StreamingTV_No internet service', 'StreamingTV_Yes',
       'StreamingMovies_No internet service', 'StreamingMovies_Yes',
       'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check'],
      dtype='object')

Rows: 7043, and columns: 30

   SeniorCitizen  tenure  MonthlyCharges  TotalCharges  gender_Male  \
0              0  

### 4.2 Train/test split
Hold out 20% to validate the model isn't just memorising training data.

In [10]:
X = model_df
y = df["Churn_binary"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {X_train.shape}, and Test: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.3f}")
print(f"Test churn rate: {y_test.mean():.3f}")

Train: (5634, 30), and Test: (1409, 30)
Train churn rate: 0.265
Test churn rate: 0.265


### 4.3 Fit the model

In [11]:
model = LogisticRegression(max_iter=2500, random_state=42)
model.fit(X_train, y_train)

train_accuracy = model.score(X_train, y_train)
test_accuracy = model.score(X_test, y_test)
print(f"Train accuracy: {train_accuracy:.3f}")
print(f"Test accuracy: {test_accuracy:.3f}")

Train accuracy: 0.806
Test accuracy: 0.806


### 4.4 Interpret coefficients
Rank features by impact; positive = increases churn risk, negative = decreases it.

In [12]:
coefficients = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": model.coef_[0],
    "odds_ratio": np.exp(model.coef_[0]),
})
coefficients["abs_coef"] = coefficients["coefficient"].abs()
coefficients = coefficients.sort_values(by="abs_coef", ascending=False).drop(columns="abs_coef")

print(coefficients.to_string(index=False))

                              feature  coefficient  odds_ratio
                    Contract_Two year    -1.339868    0.261880
          InternetService_Fiber optic     1.274647    3.577440
                    Contract_One year    -0.689151    0.502002
                      StreamingTV_Yes     0.412288    1.510270
                  StreamingMovies_Yes     0.409259    1.505701
                    MultipleLines_Yes     0.382851    1.466459
       PaymentMethod_Electronic check     0.381761    1.464861
                 PaperlessBilling_Yes     0.372007    1.450643
                   OnlineSecurity_Yes    -0.331155    0.718094
                      TechSupport_Yes    -0.279535    0.756135
       MultipleLines_No phone service     0.279506    1.322477
                       Dependents_Yes    -0.226687    0.797171
      TechSupport_No internet service    -0.185982    0.830289
  StreamingMovies_No internet service    -0.185982    0.830289
 DeviceProtection_No internet service    -0.185982    0

In [15]:
coefficients.to_csv("../outputs/model_coefficients.csv", index=False)